In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/sample_submission.csv
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/train.csv
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test.csv
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_49.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_90.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_77.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_66.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_54.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_42.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_81.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_72.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final

In [19]:
from pathlib import Path
import pandas as pd
from IPython.display import display
import numpy as np

DATA_DIR = Path(
    "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"
)

# Load the original CSVs.
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

for name, frame in [
    ("TRAIN", train_df),
    ("TEST", test_df),
    ("SAMPLE SUBMISSION", sample_submission),
]:
    print("\n" + "=" * 60)
    print(name)
    print("Shape:", frame.shape)
    print("Columns:", frame.columns.tolist())

    display(frame.head())

    print("Column types and missing values:")
    display(pd.DataFrame({
        "dtype": frame.dtypes.astype(str),
        "missing": frame.isna().sum(),
        "unique_values": frame.nunique(),
    }))

print("\nDataset folder contents:")
for path in sorted(DATA_DIR.iterdir()):
    print(f"{'FOLDER' if path.is_dir() else 'FILE'}: {path.name}")


TRAIN
Shape: (769, 2)
Columns: ['filename', 'label']


,filename,label
0,audio_192.wav,3.0
1,audio_436.wav,3.0
2,audio_447.wav,3.5
3,audio_126.wav,2.0
4,audio_61.wav,2.0


Column types and missing values:


,dtype,missing,unique_values
filename,object,0,769
label,float64,0,10



TEST
Shape: (216, 2)
Columns: ['filename', 'label']


...


SAMPLE SUBMISSION
Shape: (204, 2)
Columns: ['filename', 'label']


,filename,label
0,audio_804.wav,-1
1,audio_1028.wav,-1
2,audio_865.wav,-1
3,audio_774.wav,-1
4,audio_1138.wav,-1


Column types and missing values:


,dtype,missing,unique_values
filename,object,0,204
label,int64,0,1



Dataset folder contents:
FILE: sample_submission.csv
FOLDER: test
FILE: test.csv
FOLDER: train
FILE: train.csv


# Label , duplicate checks

In [3]:

DATA_DIR = Path(
    "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"
)

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

TRAIN_AUDIO_DIR = DATA_DIR / "train"
TEST_AUDIO_DIR = DATA_DIR / "test"

# 1. Validate training labels.
labels = pd.to_numeric(train_df["label"], errors="coerce")

invalid_labels = (
    labels.isna()
    | ~np.isfinite(labels)
    | ~labels.between(0, 5)
)

print("INVALID TRAINING LABELS:", int(invalid_labels.sum()))

print("\nTRAINING LABEL DISTRIBUTION:")
display(
    labels.value_counts()
    .sort_index()
    .rename_axis("label")
    .reset_index(name="count")
)

# 2. Check filenames before constructing paths.
for name, frame in [("train", train_df), ("test", test_df)]:
    assert frame["filename"].notna().all(), f"Missing filenames in {name}"
    assert not frame["filename"].duplicated().any(), (
        f"Duplicate filenames in {name}"
    )

# Each split uses its own audio directory.
train_df["audio_path"] = train_df["filename"].map(
    lambda filename: str(TRAIN_AUDIO_DIR / filename)
)
test_df["audio_path"] = test_df["filename"].map(
    lambda filename: str(TEST_AUDIO_DIR / filename)
)

# 3. Check listed files exist and are not empty.
for name, frame in [("TRAIN", train_df), ("TEST", test_df)]:
    missing = []
    empty = []

    for filename, audio_path in zip(frame["filename"], frame["audio_path"]):
        path = Path(audio_path)

        if not path.is_file():
            missing.append(filename)
        elif path.stat().st_size == 0:
            empty.append(filename)

    print(f"\n{name}:")
    print("Listed recordings:", len(frame))
    print("Missing files:", len(missing))
    print("Empty files:", len(empty))

    if missing:
        print("Missing examples:", missing[:10])
    if empty:
        print("Empty examples:", empty[:10])

# 4. Compare identifiers.
train_ids = set(train_df["filename"])
test_ids = set(test_df["filename"])
sample_ids = set(sample_submission["filename"])

print("\nIDENTIFIER COMPARISON:")
print("Train/test shared filenames:", len(train_ids & test_ids))
print("Test/sample shared filenames:", len(test_ids & sample_ids))
print("Test filenames absent from sample:", len(test_ids - sample_ids))
print("Sample filenames absent from test:", len(sample_ids - test_ids))

print("\nExamples in test but absent from sample:")
print(sorted(test_ids - sample_ids)[:10])

print("\nExamples in sample but absent from test:")
print(sorted(sample_ids - test_ids)[:10])

# 5. Check GPU availability for the upcoming embedding extraction.
print("\nGPU:")
try:
    import torch

    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()):
            print(f"GPU {index}: {torch.cuda.get_device_name(index)}")
    else:
        print("CUDA unavailable. Data checks can still run on CPU.")
except ImportError:
    print("PyTorch is not installed.")

INVALID TRAINING LABELS: 0

TRAINING LABEL DISTRIBUTION:


,label,count
0,0.0,37
1,1.0,1
2,1.5,3
3,2.0,102
4,2.5,79
5,3.0,174
6,3.5,64
7,4.0,124
8,4.5,52
9,5.0,133



TRAIN:
Listed recordings: 769
Missing files: 0
Empty files: 0

TEST:
Listed recordings: 216
Missing files: 0
Empty files: 0

IDENTIFIER COMPARISON:
Train/test shared filenames: 212
Test/sample shared filenames: 25
Test filenames absent from sample: 191
Sample filenames absent from test: 179

Examples in test but absent from sample:
['audio_0.wav', 'audio_1.wav', 'audio_10.wav', 'audio_100.wav', 'audio_101.wav', 'audio_102.wav', 'audio_103.wav', 'audio_104.wav', 'audio_105.wav', 'audio_106.wav']

Examples in sample but absent from test:
['audio_1001.wav', 'audio_1006.wav', 'audio_1011.wav', 'audio_1025.wav', 'audio_1028.wav', 'audio_1039.wav', 'audio_1041.wav', 'audio_1046.wav', 'audio_1055.wav', 'audio_1059.wav']

GPU:
GPU 0: Tesla T4
GPU 1: Tesla T4


# Decoding and checking recordings

In [5]:
from collections import Counter
import hashlib
import shutil
import subprocess

DATA_DIR = Path(
    "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"
)

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

# FFmpeg handles different audio formats, including files whose
# actual format may not match their extension.
FFMPEG = shutil.which("ffmpeg")

if FFMPEG is None:
    raise RuntimeError(
        "FFmpeg is unavailable. Share this error before proceeding."
    )

SAMPLE_RATE = 16000


def decode_audio(path):
    """Decode the entire recording to mono, 16 kHz, signed 16-bit PCM."""
    result = subprocess.run(
        [
            FFMPEG,
            "-nostdin",
            "-v", "error",
            "-i", str(path),
            "-vn",
            "-ac", "1",
            "-ar", str(SAMPLE_RATE),
            "-acodec", "pcm_s16le",
            "-f", "s16le",
            "pipe:1",
        ],
        capture_output=True,
        timeout=120,
    )

    if result.returncode != 0:
        message = result.stderr.decode("utf-8", errors="replace")
        raise RuntimeError(message[:500])

    if not result.stdout:
        raise ValueError("Decoder returned no audio samples.")

    pcm = np.frombuffer(result.stdout, dtype="<i2")
    audio = pcm.astype(np.float32) / 32768.0

    return audio, result.stdout


records = []
total = len(train_df) + len(test_df)

for split, frame in [("train", train_df), ("test", test_df)]:
    for row in frame.itertuples(index=False):
        path = DATA_DIR / split / row.filename

        record = {
            "split": split,
            "filename": row.filename,
            "audio_path": str(path),
            # Test labels are placeholders, not ground truth.
            "label": float(row.label) if split == "train" else np.nan,
            "decode_ok": False,
            "error": "",
        }

        try:
            audio, pcm_bytes = decode_audio(path)

            record.update({
                "decode_ok": True,
                "duration_sec": len(audio) / SAMPLE_RATE,
                "rms": float(np.sqrt(np.mean(audio ** 2))),
                "peak": float(np.max(np.abs(audio))),
                "zero_signal": bool(np.all(audio == 0)),
                # Quiet amplitude is a diagnostic, not a speech detector.
                "quiet_fraction": float(np.mean(np.abs(audio) < 0.001)),
                "audio_hash": hashlib.sha256(pcm_bytes).hexdigest(),
            })

        except Exception as exc:
            record["error"] = str(exc)[:500]

        records.append(record)

        if len(records) % 100 == 0 or len(records) == total:
            print(f"Checked {len(records)}/{total} recordings")

audio_audit = pd.DataFrame(records)
valid_audio = audio_audit.loc[audio_audit["decode_ok"]].copy()

print("\nDECODE SUMMARY:")
display(
    audio_audit.groupby("split")["decode_ok"]
    .agg(total="size", decoded="sum")
)

failed_audio = audio_audit.loc[~audio_audit["decode_ok"]]

if not failed_audio.empty:
    print("\nFILES THAT FAILED:")
    display(failed_audio[["split", "filename", "error"]])

print("\nDURATION SUMMARY — SECONDS:")
display(valid_audio.groupby("split")["duration_sec"].describe())

print("\nZERO-SIGNAL RECORDINGS:")
display(
    valid_audio.loc[
        valid_audio["zero_signal"],
        ["split", "filename", "label", "duration_sec"]
    ]
)

print("\nRECORDINGS OUTSIDE THE DESCRIBED 45–60 SECOND RANGE:")
outside_range = valid_audio.loc[
    ~valid_audio["duration_sec"].between(45, 60)
]
print("Count:", len(outside_range))
display(
    outside_range[
        ["split", "filename", "label", "duration_sec"]
    ].head(15)
)

# Exact matches after standardizing sample rate, channel count and PCM format.
duplicate_audio = valid_audio.loc[
    valid_audio.duplicated("audio_hash", keep=False)
].sort_values(["audio_hash", "split", "filename"])

print("\nEXACT STANDARDIZED-AUDIO DUPLICATES:")
print("Recordings involved:", len(duplicate_audio))
print("Duplicate groups:", duplicate_audio["audio_hash"].nunique())

display(
    duplicate_audio[
        ["split", "filename", "label", "duration_sec", "audio_hash"]
    ].head(30)
)

train_hashes = set(
    valid_audio.loc[valid_audio["split"] == "train", "audio_hash"]
)
test_hashes = set(
    valid_audio.loc[valid_audio["split"] == "test", "audio_hash"]
)

print("\nEXACT AUDIO HASHES SHARED BETWEEN TRAIN AND TEST:")
print(len(train_hashes & test_hashes))

# Check whether duplicate training audio has inconsistent labels.
train_valid = valid_audio.loc[valid_audio["split"] == "train"]
labels_per_hash = train_valid.groupby("audio_hash")["label"].nunique()
conflicting_hashes = labels_per_hash[labels_per_hash > 1].index

print("\nDUPLICATE TRAINING GROUPS WITH CONFLICTING LABELS:")
print(len(conflicting_hashes))

if len(conflicting_hashes):
    display(
        train_valid.loc[
            train_valid["audio_hash"].isin(conflicting_hashes),
            ["filename", "label", "audio_hash"]
        ].sort_values("audio_hash")
    )

Checked 100/985 recordings
Checked 200/985 recordings
Checked 300/985 recordings
Checked 400/985 recordings
Checked 500/985 recordings
Checked 600/985 recordings
Checked 700/985 recordings
Checked 800/985 recordings
Checked 900/985 recordings
Checked 985/985 recordings

DECODE SUMMARY:


,total,decoded
split,,
test,216,216
train,769,769



DURATION SUMMARY — SECONDS:


,count,mean,std,min,25%,50%,75%,max
split,,,,,,,,
test,216.0,48.755341,8.181578,6.48,45.055000,45.060000,60.010688,61.034687
train,769.0,55.781067,7.908964,20.06,54.186687,60.074688,60.074688,61.040000



ZERO-SIGNAL RECORDINGS:


...


RECORDINGS OUTSIDE THE DESCRIBED 45–60 SECOND RANGE:
Count: 700


,split,filename,label,duration_sec
0,train,audio_192.wav,3.0,60.508375
1,train,audio_436.wav,3.0,60.074688
2,train,audio_447.wav,3.5,60.074688
5,train,audio_639.wav,2.5,60.074688
7,train,audio_509.wav,3.0,60.074688
8,train,audio_672.wav,4.0,60.074688
9,train,audio_344.wav,4.0,60.074688
10,train,audio_592.wav,5.0,60.074688
11,train,audio_705.wav,3.0,60.074688
12,train,audio_497.wav,2.0,60.074688



EXACT STANDARDIZED-AUDIO DUPLICATES:
Recordings involved: 0
Duplicate groups: 0


,split,filename,label,duration_sec,audio_hash



EXACT AUDIO HASHES SHARED BETWEEN TRAIN AND TEST:
0

DUPLICATE TRAINING GROUPS WITH CONFLICTING LABELS:
0


# Baseling and creating validation sets

In [6]:
from sklearn.model_selection import StratifiedKFold


# This cell uses audio_audit from your successful audio-check cell.
if "audio_audit" not in globals():
    raise RuntimeError("Run the audio-audit cell first.")

SEED = 42
N_FOLDS = 5

OUTPUT_DIR = Path("/kaggle/working/shl_grammar")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path(
    "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"
)

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

# Merge within each split: filenames overlap between train and test.
audit_columns = [
    "filename",
    "audio_path",
    "decode_ok",
    "duration_sec",
    "audio_hash",
]

train_meta = audio_audit.loc[
    audio_audit["split"].eq("train"), audit_columns
]
test_meta = audio_audit.loc[
    audio_audit["split"].eq("test"), audit_columns
]

train_df = train_df.merge(
    train_meta,
    on="filename",
    how="left",
    validate="one_to_one",
    sort=False,
)

test_df = test_df.merge(
    test_meta,
    on="filename",
    how="left",
    validate="one_to_one",
    sort=False,
)

for name, frame in [("train", train_df), ("test", test_df)]:
    assert frame["decode_ok"].eq(True).all(), f"Audio audit missing for {name}"
    assert frame["audio_hash"].notna().all()
    assert frame["filename"].is_unique

# Exact duplicates would require grouped folds.
assert train_df["audio_hash"].is_unique

# Broader bins are ONLY used to distribute examples across folds.
train_df["score_bin"] = pd.cut(
    train_df["label"],
    bins=[-np.inf, 0.5, 2.0, 3.0, 4.0, np.inf],
    labels=["0", "1–2", "2.5–3", "3.5–4", "4.5–5"],
).astype(str)

bin_counts = train_df["score_bin"].value_counts()
assert bin_counts.min() >= N_FOLDS

splitter = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED,
)

train_df["fold"] = -1

for fold, (_, valid_idx) in enumerate(
    splitter.split(train_df, train_df["score_bin"])
):
    train_df.loc[valid_idx, "fold"] = fold

assert train_df["fold"].between(0, N_FOLDS - 1).all()

print("FOLD SUMMARY:")
display(
    train_df.groupby("fold").agg(
        recordings=("filename", "size"),
        label_mean=("label", "mean"),
        label_std=("label", "std"),
        mean_duration_sec=("duration_sec", "mean"),
    )
)

print("\nSCORE BINS BY FOLD:")
display(pd.crosstab(train_df["score_bin"], train_df["fold"]))

# Baseline: predict the mean of the TRAINING portion of each fold.
y = train_df["label"].to_numpy(dtype=np.float64)
baseline_oof = np.full(len(train_df), np.nan)
baseline_results = []

for fold in range(N_FOLDS):
    train_mask = train_df["fold"].ne(fold).to_numpy()
    valid_mask = train_df["fold"].eq(fold).to_numpy()

    prediction = y[train_mask].mean()
    baseline_oof[valid_mask] = prediction

    fold_rmse = np.sqrt(
        np.mean((y[valid_mask] - prediction) ** 2)
    )
    fold_mae = np.mean(
        np.abs(y[valid_mask] - prediction)
    )

    baseline_results.append({
        "fold": fold,
        "predicted_mean": prediction,
        "rmse": fold_rmse,
        "mae": fold_mae,
    })

assert np.isfinite(baseline_oof).all()

print("\nMEAN-PREDICTION BASELINE:")
display(pd.DataFrame(baseline_results))

overall_rmse = np.sqrt(np.mean((y - baseline_oof) ** 2))
overall_mae = np.mean(np.abs(y - baseline_oof))

print(f"Overall OOF RMSE: {overall_rmse:.4f}")
print(f"Overall OOF MAE:  {overall_mae:.4f}")
print("Per-fold Pearson is undefined for constant predictions.")

# Save the fixed assignments and audit for subsequent stages.
train_df.to_csv(OUTPUT_DIR / "train_folds.csv", index=False)

# Exclude placeholder test labels from the saved inference manifest.
test_df.drop(columns=["label"], errors="ignore").to_csv(
    OUTPUT_DIR / "test_manifest.csv",
    index=False,
)

audio_audit.to_csv(OUTPUT_DIR / "audio_audit.csv", index=False)

baseline_predictions = train_df[
    ["filename", "label", "fold"]
].copy()
baseline_predictions["prediction"] = baseline_oof
baseline_predictions.to_csv(
    OUTPUT_DIR / "baseline_oof.csv",
    index=False,
)

print("\nSaved files to:", OUTPUT_DIR)

FOLD SUMMARY:


,recordings,label_mean,label_std,mean_duration_sec
fold,,,,
0,154,3.327922,1.235874,56.243220
1,154,3.327922,1.234551,54.648382
2,154,3.314935,1.251080,56.020558
3,154,3.275974,1.248584,56.154294
4,153,3.320261,1.240244,55.839258



SCORE BINS BY FOLD:


fold,0,1,2,3,4
score_bin,,,,,
0,7,7,8,8,7
1–2,21,22,21,21,21
2.5–3,51,51,51,50,50
3.5–4,38,37,37,38,38
4.5–5,37,37,37,37,37



MEAN-PREDICTION BASELINE:


,fold,predicted_mean,rmse,mae
0,0,3.309756,1.231989,0.978825
1,1,3.309756,1.230671,0.989341
2,2,3.313008,1.247013,1.002455
3,3,3.322764,1.245403,1.011630
4,4,3.311688,1.236214,0.999576


Overall OOF RMSE: 1.2383
Overall OOF MAE:  0.9964
Per-fold Pearson is undefined for constant predictions.

Saved files to: /kaggle/working/shl_grammar


# Results : folds are balanced, and the baseline RMSE is 1.2383.

# Now will be using : WavLM-base-plus 

In [10]:
import json
import torch
from transformers import AutoFeatureExtractor, WavLMModel

# Locate a complete WavLM checkpoint among attached datasets.
candidates = []

for config_path in Path("/kaggle/input").rglob("config.json"):
    folder = config_path.parent

    if not (folder / "preprocessor_config.json").is_file():
        continue
    if not (folder / "pytorch_model.bin").is_file():
        continue

    try:
        config = json.loads(config_path.read_text())
    except (OSError, ValueError):
        continue

    if config.get("model_type") == "wavlm":
        candidates.append(folder)

if len(candidates) != 1:
    raise RuntimeError(
        f"Expected one complete WavLM checkpoint, found {len(candidates)}: "
        f"{candidates}. Check that the dataset is attached and all three "
        "files are in the same folder."
    )

MODEL_DIR = candidates[0]

assert torch.cuda.is_available(), "Enable the notebook GPU."

DEVICE = torch.device("cuda:0")
MODEL_ID = "microsoft/wavlm-base-plus"
SAMPLE_RATE = 16000
CHUNK_SECONDS = 20

print("Loading from:", MODEL_DIR)
print("GPU:", torch.cuda.get_device_name(DEVICE))

feature_extractor = AutoFeatureExtractor.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
)

wavlm = WavLMModel.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
    use_safetensors=False,  # Official checkpoint downloaded as .bin
).to(DEVICE)

wavlm.eval()
wavlm.requires_grad_(False)

assert feature_extractor.sampling_rate == SAMPLE_RATE

print("\nModel loaded successfully offline.")
print("Hidden size:", wavlm.config.hidden_size)
print("Features per recording:", wavlm.config.hidden_size * 2)

Loading from: /kaggle/input/datasets/ajttai/wavlm-base-plus-offline
GPU: Tesla T4


Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]


Model loaded successfully offline.
Hidden size: 768
Features per recording: 1536


# Extraction function , recording chucks

In [12]:
import math
import time

OUTPUT_DIR = Path("/kaggle/working/shl_grammar")
train_df = pd.read_csv(OUTPUT_DIR / "train_folds.csv")
test_df = pd.read_csv(OUTPUT_DIR / "test_manifest.csv")

SAMPLE_RATE = 16000
CHUNK_SECONDS = 20
DEVICE = torch.device("cuda:0")

FFMPEG = shutil.which("ffmpeg")
assert FFMPEG is not None, "FFmpeg is required."

wavlm.eval()


def load_waveform(audio_path):
    result = subprocess.run(
        [
            FFMPEG,
            "-nostdin",
            "-v", "error",
            "-i", str(audio_path),
            "-vn",
            "-ac", "1",
            "-ar", str(SAMPLE_RATE),
            "-acodec", "pcm_s16le",
            "-f", "s16le",
            "pipe:1",
        ],
        capture_output=True,
        timeout=120,
    )

    if result.returncode != 0:
        raise RuntimeError(
            result.stderr.decode("utf-8", errors="replace")[:500]
        )

    audio = (
        np.frombuffer(result.stdout, dtype="<i2")
        .astype(np.float32) / 32768.0
    )

    if len(audio) == 0 or not np.isfinite(audio).all():
        raise ValueError(f"Invalid audio: {audio_path}")

    return audio


@torch.inference_mode()
def extract_wavlm_embedding(audio_path):
    audio = load_waveform(audio_path)

    # Equal-sized chunks avoid a tiny leftover chunk.
    max_samples = CHUNK_SECONDS * SAMPLE_RATE
    n_chunks = max(1, math.ceil(len(audio) / max_samples))
    chunks = np.array_split(audio, n_chunks)

    # Accumulate statistics on CPU in float64.
    hidden_size = wavlm.config.hidden_size
    frame_sum = np.zeros(hidden_size, dtype=np.float64)
    frame_square_sum = np.zeros(hidden_size, dtype=np.float64)
    frame_count = 0

    for chunk in chunks:
        inputs = feature_extractor(
            chunk,
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
            padding=False,
        )
        inputs = {
            key: value.to(DEVICE)
            for key, value in inputs.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            output = wavlm(**inputs)

        hidden = (
            output.last_hidden_state[0]
            .float()
            .cpu()
            .numpy()
            .astype(np.float64)
        )

        # Each chunk is unpadded, so all output frames are valid.
        frame_sum += hidden.sum(axis=0)
        frame_square_sum += np.square(hidden).sum(axis=0)
        frame_count += hidden.shape[0]

        del inputs, output, hidden

    mean = frame_sum / frame_count
    variance = np.maximum(
        frame_square_sum / frame_count - np.square(mean),
        0.0,
    )

    embedding = np.concatenate([
        mean,
        np.sqrt(variance),
    ]).astype(np.float32)

    assert embedding.shape == (2 * hidden_size,)
    assert np.isfinite(embedding).all(), (
        f"Non-finite features: {audio_path}"
    )

    return embedding

# Testing short traning recording 

In [13]:
examples = [
    (
        "train_short",
        train_df.loc[train_df["duration_sec"].idxmin()],
    ),
    (
        "train_long",
        train_df.loc[train_df["duration_sec"].idxmax()],
    ),
    (
        "test",
        test_df.iloc[0],
    ),
]

smoke_embeddings = {}
results = []

torch.cuda.reset_peak_memory_stats(DEVICE)

for name, row in examples:
    torch.cuda.synchronize(DEVICE)
    start = time.perf_counter()

    embedding = extract_wavlm_embedding(row["audio_path"])

    torch.cuda.synchronize(DEVICE)
    elapsed = time.perf_counter() - start

    # Full paths distinguish overlapping train/test filenames.
    smoke_embeddings[row["audio_path"]] = embedding

    results.append({
        "example": name,
        "filename": row["filename"],
        "duration_sec": round(row["duration_sec"], 2),
        "features": embedding.size,
        "all_finite": bool(np.isfinite(embedding).all()),
        "extraction_sec": round(elapsed, 2),
    })

display(pd.DataFrame(results))

print(
    "Peak allocated GPU memory:",
    round(torch.cuda.max_memory_allocated(DEVICE) / 1024**3, 2),
    "GB",
)


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


,example,filename,duration_sec,features,all_finite,extraction_sec
0,train_short,audio_251.wav,20.06,1536,True,1.91
1,train_long,audio_157.wav,61.04,1536,True,0.57
2,test,audio_128.wav,45.06,1536,True,0.39


Peak allocated GPU memory: 0.6 GB


# Saving Embeddings

In [14]:
from tqdm.auto import tqdm

OUTPUT_DIR = Path("/kaggle/working/shl_grammar")
CACHE_DIR = OUTPUT_DIR / "wavlm_base_plus_lastlayer_mean_std_v1"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(OUTPUT_DIR / "train_folds.csv")
test_df = pd.read_csv(OUTPUT_DIR / "test_manifest.csv")

EXPECTED_DIM = 2 * wavlm.config.hidden_size

# Prevent accidental reuse with a different extraction configuration.
cache_config = {
    "model": "microsoft/wavlm-base-plus",
    "sample_rate": SAMPLE_RATE,
    "chunk_seconds": CHUNK_SECONDS,
    "chunking": "equal_nonoverlapping_chunks",
    "layer": "last_hidden_state",
    "pooling": "global_frame_mean_and_population_std",
    "inference_precision": "float16_autocast",
    "feature_dim": EXPECTED_DIM,
}

config_path = CACHE_DIR / "extraction_config.json"

if config_path.exists():
    saved_config = json.loads(config_path.read_text())
    if saved_config != cache_config:
        raise RuntimeError(
            "Cache configuration differs. Use a new CACHE_DIR."
        )
else:
    config_path.write_text(json.dumps(cache_config, indent=2))


def valid_embedding(array):
    return (
        array.shape == (EXPECTED_DIM,)
        and np.isfinite(array).all()
    )


def extract_split(frame, split):
    split_cache = CACHE_DIR / split
    split_cache.mkdir(parents=True, exist_ok=True)

    embeddings = []
    failures = []
    cached_count = 0
    extracted_count = 0

    rows = frame.to_dict("records")

    for row in tqdm(rows, desc=f"WavLM: {split}"):
        # Include the audited audio hash to distinguish changed recordings.
        cache_file = split_cache / (
            f"{row['filename']}.{row['audio_hash']}.npy"
        )

        try:
            embedding = None

            if cache_file.exists():
                try:
                    candidate = np.load(
                        cache_file,
                        allow_pickle=False,
                    )
                    if valid_embedding(candidate):
                        embedding = candidate.astype(np.float32)
                        cached_count += 1
                except (OSError, ValueError, EOFError):
                    # Recompute an incomplete or unreadable cache entry.
                    pass

            if embedding is None:
                # Reuse the three successful smoke-test embeddings.
                smoke_cache = globals().get("smoke_embeddings", {})
                candidate = smoke_cache.get(row["audio_path"])

                if candidate is not None and valid_embedding(candidate):
                    embedding = candidate.astype(np.float32)
                else:
                    embedding = extract_wavlm_embedding(
                        row["audio_path"]
                    )

                if not valid_embedding(embedding):
                    raise ValueError("Invalid embedding shape or values.")

                # Atomic replacement avoids leaving a partial .npy file.
                temporary_file = cache_file.with_suffix(".tmp")
                with temporary_file.open("wb") as handle:
                    np.save(
                        handle,
                        embedding,
                        allow_pickle=False,
                    )
                temporary_file.replace(cache_file)
                extracted_count += 1

            embeddings.append(embedding)

        except Exception as exc:
            failures.append({
                "split": split,
                "filename": row["filename"],
                "error": str(exc)[:500],
            })
            embeddings.append(None)

    print(
        f"{split}: {cached_count} loaded from cache, "
        f"{extracted_count} newly saved, "
        f"{len(failures)} failed"
    )

    if failures:
        failure_path = CACHE_DIR / f"{split}_failures.csv"
        pd.DataFrame(failures).to_csv(failure_path, index=False)
        display(pd.DataFrame(failures).head(10))

        raise RuntimeError(
            f"{split} extraction has failures. Completed embeddings "
            f"are cached. Details: {failure_path}"
        )

    # Stack in exactly the same order as the input manifest.
    matrix = np.stack(embeddings).astype(np.float32)

    assert matrix.shape == (len(frame), EXPECTED_DIM)
    assert np.isfinite(matrix).all()

    np.save(CACHE_DIR / f"X_{split}.npy", matrix)

    # Save row identifiers alongside the feature matrix.
    frame[["filename", "audio_hash"]].to_csv(
        CACHE_DIR / f"{split}_rows.csv",
        index=False,
    )

    return matrix


started = time.perf_counter()

X_train = extract_split(train_df, "train")
X_test = extract_split(test_df, "test")

print("\nEXTRACTION COMPLETE")
print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("All training features finite:", np.isfinite(X_train).all())
print("All test features finite:    ", np.isfinite(X_test).all())
print("Elapsed minutes:", round((time.perf_counter() - started) / 60, 2))
print("Cache directory:", CACHE_DIR)

WavLM: train:   0%|          | 0/769 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


train: 0 loaded from cache, 769 newly saved, 0 failed


WavLM: test:   0%|          | 0/216 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


test: 0 loaded from cache, 216 newly saved, 0 failed

EXTRACTION COMPLETE
X_train: (769, 1536)
X_test:  (216, 1536)
All training features finite: True
All test features finite:     True
Elapsed minutes: 7.3
Cache directory: /kaggle/working/shl_grammar/wavlm_base_plus_lastlayer_mean_std_v1


# Load features 

In [15]:
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from IPython.display import display, FileLink

OUTPUT_DIR = Path("/kaggle/working/shl_grammar")
CACHE_DIR = OUTPUT_DIR / "wavlm_base_plus_lastlayer_mean_std_v1"
RESULT_DIR = OUTPUT_DIR / "ridge_v1"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(OUTPUT_DIR / "train_folds.csv")
test_df = pd.read_csv(OUTPUT_DIR / "test_manifest.csv")

X_train = np.load(CACHE_DIR / "X_train.npy", allow_pickle=False)
X_test = np.load(CACHE_DIR / "X_test.npy", allow_pickle=False)

# Verify both filename and audio hash, preserving exact row order.
for split, frame, matrix in [
    ("train", train_df, X_train),
    ("test", test_df, X_test),
]:
    saved_rows = pd.read_csv(CACHE_DIR / f"{split}_rows.csv")

    pd.testing.assert_frame_equal(
        frame[["filename", "audio_hash"]].reset_index(drop=True),
        saved_rows[["filename", "audio_hash"]].reset_index(drop=True),
    )

    assert matrix.shape == (len(frame), 1536)
    assert np.isfinite(matrix).all()

y = train_df["label"].to_numpy(dtype=np.float64)
fold_ids = train_df["fold"].to_numpy()
folds = sorted(np.unique(fold_ids))

assert len(folds) == 5
assert np.isfinite(y).all()
assert ((y >= 0) & (y <= 5)).all()

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)
print("Row alignment verified.")

Training features: (769, 1536)
Test features: (216, 1536)
Row alignment verified.


Compare Ridge settings 

In [16]:
ALPHAS = [0.1, 1.0, 10.0, 100.0, 1000.0]


def calculate_metrics(actual, predicted):
    rmse = float(np.sqrt(np.mean((actual - predicted) ** 2)))
    mae = float(np.mean(np.abs(actual - predicted)))

    if np.std(actual) == 0 or np.std(predicted) == 0:
        pearson = np.nan
    else:
        pearson = float(np.corrcoef(actual, predicted)[0, 1])

    return {
        "rmse": rmse,
        "mae": mae,
        "pearson": pearson,
    }


results = []
fold_results = []
oof_by_alpha = {}
test_by_alpha = {}
models_by_alpha = {}

for alpha in ALPHAS:
    oof_raw = np.full(len(train_df), np.nan)
    test_fold_predictions = []
    fitted_models = []

    for fold in folds:
        training_mask = fold_ids != fold
        validation_mask = fold_ids == fold

        model = make_pipeline(
            StandardScaler(),
            Ridge(
                alpha=alpha,
                solver="lsqr",
                tol=1e-5,
                max_iter=10000,
            ),
        )

        model.fit(
            X_train[training_mask],
            y[training_mask],
        )

        valid_predictions = model.predict(X_train[validation_mask])
        oof_raw[validation_mask] = valid_predictions

        test_fold_predictions.append(model.predict(X_test))
        fitted_models.append(model)

        metrics = calculate_metrics(
            y[validation_mask],
            np.clip(valid_predictions, 0, 5),
        )

        fold_results.append({
            "alpha": alpha,
            "fold": int(fold),
            **metrics,
        })

    assert np.isfinite(oof_raw).all()

    oof_clipped = np.clip(oof_raw, 0, 5)
    metrics = calculate_metrics(y, oof_clipped)

    results.append({
        "alpha": alpha,
        "rmse_raw": calculate_metrics(y, oof_raw)["rmse"],
        "rmse": metrics["rmse"],
        "mae": metrics["mae"],
        "pearson": metrics["pearson"],
    })

    oof_by_alpha[alpha] = oof_raw
    test_by_alpha[alpha] = np.mean(test_fold_predictions, axis=0)
    models_by_alpha[alpha] = fitted_models

    print(
        f"alpha={alpha:<7g} "
        f"OOF RMSE={metrics['rmse']:.4f}  "
        f"Pearson={metrics['pearson']:.4f}"
    )

results_df = pd.DataFrame(results).sort_values(
    "rmse"
).reset_index(drop=True)

fold_results_df = pd.DataFrame(fold_results)

print("\nMODEL COMPARISON — clipped predictions:")
display(results_df)

best_alpha = float(results_df.iloc[0]["alpha"])

print("\nSelected alpha:", best_alpha)
print("\nSELECTED MODEL — PER-FOLD RESULTS:")
display(
    fold_results_df.loc[
        fold_results_df["alpha"].eq(best_alpha)
    ].reset_index(drop=True)
)

alpha=0.1     OOF RMSE=0.8367  Pearson=0.7642
alpha=1       OOF RMSE=0.8226  Pearson=0.7710
alpha=10      OOF RMSE=0.7460  Pearson=0.8077
alpha=100     OOF RMSE=0.6278  Pearson=0.8622
alpha=1000    OOF RMSE=0.5961  Pearson=0.8783

MODEL COMPARISON — clipped predictions:


,alpha,rmse_raw,rmse,mae,pearson
0,1000.0,0.597151,0.596120,0.461835,0.878273
1,100.0,0.634022,0.627771,0.477296,0.862241
2,10.0,0.775933,0.746003,0.565370,0.807732
3,1.0,0.871757,0.822572,0.622647,0.770991
4,0.1,0.889471,0.836680,0.632668,0.764203



Selected alpha: 1000.0

SELECTED MODEL — PER-FOLD RESULTS:


,alpha,fold,rmse,mae,pearson
0,1000.0,0,0.608732,0.467782,0.871810
1,1000.0,1,0.640493,0.501417,0.855682
2,1000.0,2,0.532688,0.414757,0.907142
3,1000.0,3,0.612110,0.468356,0.872361
4,1000.0,4,0.580910,0.456830,0.886218


# Saving predictions & trained pipelines

In [17]:
best_oof_raw = oof_by_alpha[best_alpha]
best_oof = np.clip(best_oof_raw, 0, 5)

best_test_raw = test_by_alpha[best_alpha]
best_test = np.clip(best_test_raw, 0, 5)

# Calculate the fold-mean baseline using the same folds.
baseline_oof = np.zeros(len(y), dtype=np.float64)

for fold in folds:
    baseline_oof[fold_ids == fold] = y[fold_ids != fold].mean()

baseline_rmse = calculate_metrics(y, baseline_oof)["rmse"]
best_metrics = calculate_metrics(y, best_oof)

print(f"Baseline OOF RMSE: {baseline_rmse:.4f}")
print(f"WavLM Ridge RMSE:  {best_metrics['rmse']:.4f}")
print(f"WavLM Ridge Pearson: {best_metrics['pearson']:.4f}")
print(
    "RMSE reduction:",
    f"{100 * (1 - best_metrics['rmse'] / baseline_rmse):.1f}%"
)

# Save validation predictions for error analysis and later blending.
oof_output = train_df[
    ["filename", "label", "fold", "duration_sec"]
].copy()

oof_output["prediction_raw"] = best_oof_raw
oof_output["prediction"] = best_oof
oof_output["absolute_error"] = np.abs(y - best_oof)

oof_output.to_csv(RESULT_DIR / "oof_predictions.csv", index=False)
results_df.to_csv(RESULT_DIR / "alpha_comparison.csv", index=False)
fold_results_df.to_csv(RESULT_DIR / "fold_metrics.csv", index=False)

# Preserve every candidate's OOF predictions.
all_oof = train_df[["filename", "label", "fold"]].copy()
for alpha in ALPHAS:
    all_oof[f"alpha_{alpha:g}_raw"] = oof_by_alpha[alpha]
all_oof.to_csv(RESULT_DIR / "all_oof_predictions.csv", index=False)

# Preserve the fitted scalers and regressors together.
joblib.dump(
    models_by_alpha[best_alpha],
    RESULT_DIR / "fold_pipelines.joblib",
)

(RESULT_DIR / "run_config.json").write_text(json.dumps({
    "alpha": best_alpha,
    "selection_metric": "clipped_oof_rmse",
    "test_prediction": "mean_of_five_fold_models_then_clip",
    "feature_cache": str(CACHE_DIR),
    "oof_metrics": best_metrics,
}, indent=2))

# Build predictions in test.csv order.
submission = pd.DataFrame({
    "filename": test_df["filename"],
    "label": best_test,
})

assert len(submission) == 216
assert submission["filename"].is_unique
assert submission["filename"].tolist() == test_df["filename"].tolist()
assert np.isfinite(submission["label"]).all()
assert submission["label"].between(0, 5).all()

submission_path = RESULT_DIR / "submission_wavlm_ridge.csv"
submission.to_csv(submission_path, index=False)

print("\nTEST PREDICTION SUMMARY:")
display(submission["label"].describe())

print("\nFIRST FIVE PREDICTIONS:")
display(submission.head())

print("\nSaved prediction file:")
display(FileLink(str(submission_path)))

Baseline OOF RMSE: 1.2383
WavLM Ridge RMSE:  0.5961
WavLM Ridge Pearson: 0.8783
RMSE reduction: 51.9%

TEST PREDICTION SUMMARY:


count    216.000000
mean       3.349793
std        0.690851
min        1.796119
25%        2.827252
50%        3.268784
75%        3.799925
max        5.000000
Name: label, dtype: float64


FIRST FIVE PREDICTIONS:


,filename,label
0,audio_128.wav,3.092188
1,audio_156.wav,3.801827
2,audio_19.wav,4.505434
3,audio_108.wav,2.083788
4,audio_206.wav,2.675048



Saved prediction file:


/kaggle/working/shl_grammar/ridge_v1/submission_wavlm_ridge.csv

In [18]:
from IPython.display import FileLink, display

display(FileLink(
    "/kaggle/working/shl_grammar/ridge_v1/"
    "submission_wavlm_ridge.csv"
))

/kaggle/working/shl_grammar/ridge_v1/submission_wavlm_ridge.csv